In [8]:
# cell 1
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
secret_label = "HF"
secret_value = UserSecretsClient().get_secret(secret_label)
# HF token from Kaggle secrets, if present
try:
    os.environ["HF_TOKEN"] = secret_value
    login(secret_value)
    print("HF login ok")
except Exception as e:
    print("HF secret not available:", e)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HF login ok


In [9]:
# =============================================================================
# Architecture 2: Encoder-Decoder Transformer (Seq2Seq) for Legal LM
# =============================================================================
# This notebook trains a ~61M parameter encoder-decoder model from scratch
# on the Indian legal corpus. It reuses the BPE tokenizer and data shards
# from the decoder-only baseline.

import os, sys, json, math, time, random, shutil, zipfile, pickle
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Optional, Dict, List, Tuple, Any
from collections import OrderedDict
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ---- Paths -----------------------------------------------------------------
IS_KAGGLE = Path("/kaggle/working").exists()
WORK_DIR = Path("/kaggle/working") if IS_KAGGLE else Path("./work")
WORK_DIR.mkdir(parents=True, exist_ok=True)
TOKENIZER_DIR = Path("/kaggle/input/datasets/ascyrax/capstone-tokenizer-large")
if not TOKENIZER_DIR.exists():
    # fallback if you saved it elsewhere
    TOKENIZER_DIR = Path("/kaggle/working/capstone_tokenizer")
    if not TOKENIZER_DIR.exists():
        raise FileNotFoundError("Tokenizer artifacts not found. Please upload them.")


# ---- Device ----------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_FP16 = (device.type == "cuda") and not torch.cuda.is_bf16_supported()
print(f"Device: {device} | fp16 autocast: {USE_FP16}")

# ---- Reproducibility -------------------------------------------------------
SEED = 1337
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if device.type == "cuda":
    torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# ---- Load Tokenizer --------------------------------------------------------
sys.path.append(str(TOKENIZER_DIR))
from bpe_tokenizer import BPETokenizer, load_tokenizer

tokenizer = load_tokenizer(str(TOKENIZER_DIR))
VOCAB_SIZE = tokenizer.vocab_size
BOS_ID = tokenizer.bos_id
EOS_ID = tokenizer.eos_id
PAD_ID = tokenizer.pad_id
UNK_ID = tokenizer.unk_id

print(f"Tokenizer loaded: vocab_size={VOCAB_SIZE}, BOS={BOS_ID}, EOS={EOS_ID}")


Device: cuda | fp16 autocast: False
Tokenizer loaded: vocab_size=32000, BOS=2, EOS=3


In [10]:
# =============================================================================
# DATA PREPARATION (Copy from decoder-only notebook)
# =============================================================================
!pip install -q ijson

import os, json, time, hashlib
from pathlib import Path
from array import array
import ijson
from huggingface_hub import HfFileSystem
from tqdm.auto import tqdm
import numpy as np


HF_REPO = "Exploration-Lab/CS-781-Capstone"

SOURCE_PATHS = {
    "acts":          "Phase_1_Pretraining/Train/train_acts.json",
    "high_courts":   "Phase_1_Pretraining/Train/train_high_courts.json",
    "supreme_court": "Phase_1_Pretraining/Train/train_supreme_court.json",
    "tribunals":     "Phase_1_Pretraining/Train/train_tribunals.json",
}

SHARD_DIR     = WORK_DIR / "shards"
SHARD_TOKENS  = 50_000_000
VAL_FRACTION  = 0.005
VAL_CAP_TOTAL = 5_000_000

TOKEN_CAPS = {
    "acts":          20_000_000,
    "high_courts":   65_000_000,
    "supreme_court": 25_000_000,
    "tribunals":     40_000_000,
}

def clean_text(t): 
    return (t or "").replace("\r\n", "\n").strip()

def is_val(doc_id, salt="phase1"):
    h = int(hashlib.md5((salt + "::" + doc_id).encode()).hexdigest(), 16)
    return (h % 10_000) / 10_000.0 < VAL_FRACTION

def stream_docs(path_in_repo):
    fs = HfFileSystem()
    url = f"datasets/{HF_REPO}/{path_in_repo}"
    with fs.open(url, "rb") as f:
        for i, doc in enumerate(ijson.items(f, "item")):
            if isinstance(doc, dict):
                doc.setdefault("_idx", i)
                yield doc

def encode_all():
    summary = {}
    for source, path_in_repo in SOURCE_PATHS.items():
        out_dir = SHARD_DIR / source
        out_dir.mkdir(parents=True, exist_ok=True)

        print(f"\n[{source}] streaming {path_in_repo}")
        meta = {"source": source, "hf_path": path_in_repo, "shards": [],
                "total_tokens": 0, "total_bytes": 0, "num_docs": 0}

        shard_ids = array("H")
        shard_idx = 0
        val_ids   = array("H")
        val_cap   = VAL_CAP_TOTAL // len(SOURCE_PATHS)
        cap       = TOKEN_CAPS[source]

        def flush():
            nonlocal shard_ids, shard_idx
            if len(shard_ids) == 0:
                return
            arr = np.frombuffer(shard_ids, dtype=np.uint16)
            final = out_dir / f"shard_{shard_idx:04d}.bin"
            tmp   = final.with_suffix(".bin.tmp")
            arr.tofile(tmp); tmp.rename(final)
            meta["shards"].append({"name": final.name, "tokens": int(arr.size)})
            meta["total_tokens"] += int(arr.size)
            print(f"  [{source}] {final.name}: {arr.size/1e6:.1f}M tokens")
            shard_ids = array("H")
            shard_idx += 1

        pbar = tqdm(desc=f"  {source}", unit="doc")
        try:
            for doc in stream_docs(path_in_repo):
                pbar.update(1)
                text = clean_text(doc.get("text"))
                if not text:
                    continue
                doc_id = str(doc.get("url") or doc.get("id") or doc.get("_idx"))
                meta["num_docs"] += 1
                meta["total_bytes"] += len(text.encode("utf-8"))

                ids = tokenizer.encode(text, allow_special=False)

                if is_val(doc_id) and len(val_ids) < val_cap:
                    val_ids.append(BOS_ID)
                    val_ids.extend(ids)
                    val_ids.append(EOS_ID)
                    continue

                shard_ids.append(BOS_ID)
                shard_ids.extend(ids)
                shard_ids.append(EOS_ID)

                if len(shard_ids) >= SHARD_TOKENS:
                    flush()
                if meta["total_tokens"] + len(shard_ids) >= cap:
                    print(f"  [{source}] hit cap {cap/1e6:.0f}M")
                    break
        finally:
            pbar.close()

        flush()
        if len(val_ids) > 0:
            arr = np.frombuffer(val_ids, dtype=np.uint16)
            (out_dir / "val.bin").write_bytes(arr.tobytes())
            meta["val_tokens"] = int(arr.size)

        with open(out_dir / "meta.json", "w") as f:
            json.dump(meta, f, indent=2)
        summary[source] = meta
        print(f"[{source}] {meta['total_tokens']/1e6:.1f}M tokens, "
              f"{meta['num_docs']} docs, val={meta.get('val_tokens',0)/1e6:.2f}M\n")
    return summary

# Run the preparation
# Note: This will download and tokenize ~150M tokens. Takes about 3-5 mins.
results = encode_all()
print("Data preparation complete!")


[acts] streaming Phase_1_Pretraining/Train/train_acts.json


  acts: 0doc [00:00, ?doc/s]

  [acts] hit cap 20M
  [acts] shard_0000.bin: 20.0M tokens
[acts] 20.0M tokens, 2228 docs, val=0.07M


[high_courts] streaming Phase_1_Pretraining/Train/train_high_courts.json


  high_courts: 0doc [00:00, ?doc/s]

  [high_courts] shard_0000.bin: 50.0M tokens
  [high_courts] hit cap 65M
  [high_courts] shard_0001.bin: 15.0M tokens
[high_courts] 65.0M tokens, 112740 docs, val=0.31M


[supreme_court] streaming Phase_1_Pretraining/Train/train_supreme_court.json


  supreme_court: 0doc [00:00, ?doc/s]

  [supreme_court] hit cap 25M
  [supreme_court] shard_0000.bin: 25.0M tokens
[supreme_court] 25.0M tokens, 3650 docs, val=0.12M


[tribunals] streaming Phase_1_Pretraining/Train/train_tribunals.json


  tribunals: 0doc [00:00, ?doc/s]

  [tribunals] hit cap 40M
  [tribunals] shard_0000.bin: 40.0M tokens
[tribunals] 40.0M tokens, 8466 docs, val=0.20M

Data preparation complete!


In [11]:
# SHARD_DIR = WORK_DIR / "shards"
# if not SHARD_DIR.exists():
#     raise FileNotFoundError("Data shards not found. Run the data preparation cell from the decoder-only notebook first.")

In [12]:
# 1. Model Configuration
# =============================================================================
@dataclass
class ModelConfig:
    # Shape
    vocab_size: int = VOCAB_SIZE
    n_encoder_layers: int = 6
    n_decoder_layers: int = 6
    d_model: int = 512
    n_head: int = 8
    n_kv_head: int = 8
    head_dim: int = 64
    ff_mult: float = 8/3
    d_ff: Optional[int] = None
    max_seq_len: int = 1024
    encoder_len: int = 512
    decoder_len: int = 128

    # Components
    norm_type: str = "rmsnorm"
    norm_eps: float = 1e-5
    pos_type: str = "rope"
    rope_theta: float = 10000.0
    mlp_type: str = "swiglu"
    bias: bool = False
    attn_dropout: float = 0.0
    resid_dropout: float = 0.0
    embd_dropout: float = 0.0
    tie_embeddings: bool = True
    init_std: float = 0.02
    use_sdpa: bool = True

    def __post_init__(self):
        assert self.d_model == self.n_head * self.head_dim
        assert self.n_head % self.n_kv_head == 0
        if self.d_ff is None:
            raw = int(self.ff_mult * self.d_model)
            self.d_ff = 64 * ((raw + 63) // 64)


In [13]:
# 2. Core Building Blocks (reused from decoder-only)
# =============================================================================
class RMSNorm(nn.Module):
    def __init__(self, d: int, eps: float = 1e-5):
        super().__init__()
        self.g = nn.Parameter(torch.ones(d))
        self.eps = eps
    def forward(self, x):
        rms = torch.sqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps).to(x.dtype)
        return (x / rms) * self.g

class LayerNorm(nn.Module):
    def __init__(self, d: int, eps: float = 1e-5):
        super().__init__()
        self.g = nn.Parameter(torch.ones(d))
        self.b = nn.Parameter(torch.zeros(d))
        self.eps = eps
    def forward(self, x):
        mean = x.mean(-1, keepdim=True)
        var = x.var(-1, keepdim=True, unbiased=False)
        return (x - mean) / torch.sqrt(var + self.eps) * self.g + self.b

def build_norm(cfg: ModelConfig):
    if cfg.norm_type == "layernorm":
        return LayerNorm(cfg.d_model, cfg.norm_eps)
    elif cfg.norm_type == "rmsnorm":
        return RMSNorm(cfg.d_model, cfg.norm_eps)
    raise ValueError(cfg.norm_type)

class RoPE(nn.Module):
    def __init__(self, head_dim: int, max_seq_len: int, theta: float = 10000.0):
        super().__init__()
        assert head_dim % 2 == 0
        self.head_dim = head_dim
        inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))
        t = torch.arange(max_seq_len).float()
        freqs = torch.outer(t, inv_freq)
        self.register_buffer("cos", freqs.cos(), persistent=False)
        self.register_buffer("sin", freqs.sin(), persistent=False)

    @staticmethod
    def _rotate(x, cos, sin):
        x1 = x[..., 0::2]
        x2 = x[..., 1::2]
        o1 = x1 * cos - x2 * sin
        o2 = x1 * sin + x2 * cos
        return torch.stack([o1, o2], dim=-1).flatten(-2)

    def forward(self, q, k, position_ids=None):
        B, H, T, D = q.shape
        if position_ids is None:
            cos = self.cos[:T].unsqueeze(0).unsqueeze(0)
            sin = self.sin[:T].unsqueeze(0).unsqueeze(0)
        else:
            cos = self.cos[position_ids].unsqueeze(1)  # [B,1,T,D/2]
            sin = self.sin[position_ids].unsqueeze(1)
        in_dtype = q.dtype
        q = q.float(); k = k.float()
        q = self._rotate(q, cos.float(), sin.float())
        k = self._rotate(k, cos.float(), sin.float())
        return q.to(in_dtype), k.to(in_dtype)

class Attention(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        self.n_head = cfg.n_head
        self.n_kv_head = cfg.n_kv_head
        self.head_dim = cfg.head_dim
        self.scale = 1.0 / math.sqrt(self.head_dim)

        self.q_proj = nn.Linear(cfg.d_model, cfg.n_head * cfg.head_dim, bias=cfg.bias)
        self.k_proj = nn.Linear(cfg.d_model, cfg.n_kv_head * cfg.head_dim, bias=cfg.bias)
        self.v_proj = nn.Linear(cfg.d_model, cfg.n_kv_head * cfg.head_dim, bias=cfg.bias)
        self.o_proj = nn.Linear(cfg.n_head * cfg.head_dim, cfg.d_model, bias=cfg.bias)

        self.attn_dropout = nn.Dropout(cfg.attn_dropout)
        self.resid_dropout = nn.Dropout(cfg.resid_dropout)

    def _split(self, x, n_head):
        B, T, _ = x.shape
        return x.view(B, T, n_head, self.head_dim).transpose(1, 2)

    def forward(self, x, rope=None, position_ids=None, attention_mask=None,
                kv_cache=None, context=None):
        B, T, _ = x.shape
        kv_src = context if context is not None else x
        S = kv_src.shape[1]

        q = self._split(self.q_proj(x), self.n_head)
        k = self._split(self.k_proj(kv_src), self.n_kv_head)
        v = self._split(self.v_proj(kv_src), self.n_kv_head)

        if rope is not None and context is None:
            q, k = rope(q, k, position_ids=position_ids)

        if kv_cache is not None:
            k_prev, v_prev = kv_cache
            k = torch.cat([k_prev, k], dim=2)
            v = torch.cat([v_prev, v], dim=2)
        new_cache = (k, v)

        if self.n_kv_head != self.n_head:
            rep = self.n_head // self.n_kv_head
            k = k.repeat_interleave(rep, dim=1)
            v = v.repeat_interleave(rep, dim=1)

        is_causal = (context is None) and (self.cfg.causal if hasattr(self.cfg, 'causal') else True) and (T > 1 or kv_cache is None)

        pad_mask = None
        if attention_mask is not None:
            keep = attention_mask[:, None, None, :].to(torch.bool)
            pad_mask = torch.zeros(B, 1, T, S, device=x.device, dtype=torch.float32)
            pad_mask = pad_mask.masked_fill(~keep, float("-inf"))

        use_sdpa = self.cfg.use_sdpa and hasattr(F, "scaled_dot_product_attention")

        if use_sdpa and pad_mask is None and (is_causal or T == 1):
            out = F.scaled_dot_product_attention(
                q, k, v, attn_mask=None,
                dropout_p=self.attn_dropout.p if self.training else 0.0,
                is_causal=is_causal,
            )
        elif use_sdpa:
            if is_causal:
                causal = torch.triu(torch.full((T, S), float("-inf"), device=x.device), diagonal=S - T + 1)
                additive = causal.view(1, 1, T, S)
                if pad_mask is not None:
                    additive = additive + pad_mask
            else:
                additive = pad_mask
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=additive,
                                                 dropout_p=self.attn_dropout.p if self.training else 0.0,
                                                 is_causal=False)
        else:
            scores = (q @ k.transpose(-2, -1)) * self.scale
            if is_causal:
                causal = torch.triu(torch.ones(T, S, device=x.device, dtype=torch.bool), diagonal=S - T + 1)
                scores = scores.masked_fill(causal, float("-inf"))
            if pad_mask is not None:
                scores = scores + pad_mask.to(scores.dtype)
            scores = F.softmax(scores.float(), dim=-1).to(q.dtype)
            scores = self.attn_dropout(scores)
            out = scores @ v

        out = out.transpose(1, 2).contiguous().view(B, T, -1)
        return self.resid_dropout(self.o_proj(out)), new_cache

class SwiGLU(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.gate_proj = nn.Linear(cfg.d_model, cfg.d_ff, bias=cfg.bias)
        self.up_proj = nn.Linear(cfg.d_model, cfg.d_ff, bias=cfg.bias)
        self.down_proj = nn.Linear(cfg.d_ff, cfg.d_model, bias=cfg.bias)
        self.drop = nn.Dropout(cfg.resid_dropout)
    def forward(self, x):
        return self.drop(self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x)))

class GELUMLP(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        hidden = cfg.d_ff
        self.fc1 = nn.Linear(cfg.d_model, hidden, bias=cfg.bias)
        self.fc2 = nn.Linear(hidden, cfg.d_model, bias=cfg.bias)
        self.drop = nn.Dropout(cfg.resid_dropout)
    def forward(self, x):
        return self.drop(self.fc2(F.gelu(self.fc1(x))))

def build_channel_mixer(cfg: ModelConfig):
    if cfg.mlp_type == "swiglu":
        return SwiGLU(cfg)
    elif cfg.mlp_type == "gelu":
        return GELUMLP(cfg)
    raise ValueError(cfg.mlp_type)


In [14]:
# 3. Encoder-Decoder Architecture
# =============================================================================
class EncoderBlock(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.norm1 = build_norm(cfg)
        self.self_attn = Attention(cfg)
        self.norm2 = build_norm(cfg)
        self.mlp = build_channel_mixer(cfg)

    def forward(self, x, rope=None, attention_mask=None, position_ids=None):
        h, _ = self.self_attn(self.norm1(x), rope=rope, position_ids=position_ids, attention_mask=attention_mask)
        x = x + h
        x = x + self.mlp(self.norm2(x))
        return x

class DecoderBlock(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.norm1 = build_norm(cfg)
        self.self_attn = Attention(cfg)      # causal self-attention
        self.norm2 = build_norm(cfg)
        self.cross_attn = Attention(cfg)     # cross-attention
        self.norm3 = build_norm(cfg)
        self.mlp = build_channel_mixer(cfg)

    def forward(self, x, encoder_output, rope=None, attention_mask=None, cross_attention_mask=None, enc_attention_mask=None,
                position_ids=None, kv_cache=None):
        # Self-attention with causal mask
        h, new_self_cache = self.self_attn(
            self.norm1(x), rope=rope, position_ids=position_ids,
            attention_mask=attention_mask,
            kv_cache=kv_cache[0] if kv_cache else None
        )
        x = x + h
        # Cross-attention: query from x, context = encoder_output
        h_cross, _ = self.cross_attn(
            self.norm2(x), context=encoder_output, rope=None,
            # attention_mask=None  # no padding mask for encoder output
            attention_mask=enc_attention_mask  # no padding mask for encoder output
        )
        x = x + h_cross
        x = x + self.mlp(self.norm3(x))
        return x, (new_self_cache, None)

class Encoder(nn.Module):
    def __init__(self, cfg: ModelConfig, tok_emb: nn.Embedding):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = tok_emb
        self.emb_drop = nn.Dropout(cfg.embd_dropout)
        self.rope = RoPE(cfg.head_dim, cfg.max_seq_len, cfg.rope_theta) if cfg.pos_type == "rope" else None
        if cfg.pos_type == "learned":
            self.pos_emb = nn.Embedding(cfg.max_seq_len, cfg.d_model)
        else:
            self.pos_emb = None
        self.blocks = nn.ModuleList([EncoderBlock(cfg) for _ in range(cfg.n_encoder_layers)])
        self.norm_f = build_norm(cfg)

    def forward(self, input_ids, attention_mask=None, position_ids=None):
        x = self.tok_emb(input_ids)
        if self.pos_emb is not None:
            if position_ids is None:
                position_ids = torch.arange(input_ids.size(1), device=input_ids.device).unsqueeze(0)
            x = x + self.pos_emb(position_ids)
        x = self.emb_drop(x)
        for blk in self.blocks:
            x = blk(x, rope=self.rope, attention_mask=attention_mask, position_ids=position_ids)
        x = self.norm_f(x)
        return x

class Decoder(nn.Module):
    def __init__(self, cfg: ModelConfig, tok_emb: nn.Embedding):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = tok_emb
        self.emb_drop = nn.Dropout(cfg.embd_dropout)
        self.rope = RoPE(cfg.head_dim, cfg.max_seq_len, cfg.rope_theta) if cfg.pos_type == "rope" else None
        if cfg.pos_type == "learned":
            self.pos_emb = nn.Embedding(cfg.max_seq_len, cfg.d_model)
        else:
            self.pos_emb = None
        self.blocks = nn.ModuleList([DecoderBlock(cfg) for _ in range(cfg.n_decoder_layers)])
        self.norm_f = build_norm(cfg)

    def forward(self, input_ids, encoder_output, attention_mask=None, cross_attention_mask=None, enc_attention_mask=None,
                position_ids=None, kv_caches=None):
        x = self.tok_emb(input_ids)
        if self.pos_emb is not None:
            if position_ids is None:
                position_ids = torch.arange(input_ids.size(1), device=input_ids.device).unsqueeze(0)
            x = x + self.pos_emb(position_ids)
        x = self.emb_drop(x)
        new_caches = []
        for i, blk in enumerate(self.blocks):
            cache = kv_caches[i] if kv_caches is not None else None
            x, nc = blk(x, encoder_output, rope=self.rope,
                        attention_mask=attention_mask, 
                        cross_attention_mask=cross_attention_mask,
                        enc_attention_mask=enc_attention_mask,
                        position_ids=position_ids,
                        kv_cache=cache)
            new_caches.append(nc)
        x = self.norm_f(x)
        return x, new_caches

class Seq2SeqLM(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.encoder = Encoder(cfg, self.tok_emb)
        self.decoder = Decoder(cfg, self.tok_emb)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        if cfg.tie_embeddings:
            self.lm_head.weight = self.tok_emb.weight

    def forward(self, enc_input_ids, dec_input_ids,
                enc_attention_mask=None, dec_attention_mask=None,
                enc_position_ids=None, dec_position_ids=None,
                kv_caches=None):
        encoder_output = self.encoder(enc_input_ids, attention_mask=enc_attention_mask,
                                      position_ids=enc_position_ids)
        decoder_output, new_caches = self.decoder(
            dec_input_ids, encoder_output,
            attention_mask=dec_attention_mask,
            enc_attention_mask=enc_attention_mask,
            position_ids=dec_position_ids,
            kv_caches=kv_caches
        )
        logits = self.lm_head(decoder_output)
        return logits, new_caches


In [15]:
# 4. Weight Initialization
# =============================================================================
def init_weights(model: Seq2SeqLM, cfg: ModelConfig):
    for name, m in model.named_modules():
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=cfg.init_std)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=cfg.init_std)

    scale = 1.0 / math.sqrt(2 * max(cfg.n_encoder_layers, cfg.n_decoder_layers))
    for blk in model.encoder.blocks:
        if hasattr(blk.self_attn, "o_proj"):
            with torch.no_grad(): blk.self_attn.o_proj.weight.mul_(scale)
        if hasattr(blk.mlp, "down_proj"):
            with torch.no_grad(): blk.mlp.down_proj.weight.mul_(scale)
    for blk in model.decoder.blocks:
        if hasattr(blk.self_attn, "o_proj"):
            with torch.no_grad(): blk.self_attn.o_proj.weight.mul_(scale)
        if hasattr(blk.cross_attn, "o_proj"):
            with torch.no_grad(): blk.cross_attn.o_proj.weight.mul_(scale)
        if hasattr(blk.mlp, "down_proj"):
            with torch.no_grad(): blk.mlp.down_proj.weight.mul_(scale)

def count_params(model: nn.Module) -> Dict[str, int]:
    emb = model.tok_emb.weight.numel()
    seen = set()
    def _sum(mod):
        n = 0
        for p in mod.parameters():
            if id(p) in seen: continue
            seen.add(id(p))
            n += p.numel()
        return n
    total = _sum(model)
    return {"total": total, "embedding": emb, "non_embedding": total - emb}


In [16]:
# 5. Data Loading for Seq2Seq
# =============================================================================
# We reuse the shards but sample windows of length encoder_len + decoder_len + 1
class ShardReader:
    def __init__(self, source_dir: Path):
        self.dir = Path(source_dir)
        meta = json.loads((self.dir / "meta.json").read_text())
        self.shards = meta["shards"]
        self.total_tokens = sum(s["tokens"] for s in self.shards)
        self.arrays = [np.memmap(self.dir / s["name"], dtype=np.uint16, mode="r")
                       for s in self.shards]
        self.shard_weights = np.array([s["tokens"] for s in self.shards], dtype=np.float64)
        self.shard_weights /= self.shard_weights.sum()
        self.val_path = self.dir / "val.bin"
        self.val_arr = np.memmap(self.val_path, dtype=np.uint16, mode="r") if self.val_path.exists() else None

class Seq2SeqWindowLoader:
    def __init__(self, sources: Dict[str, ShardReader], mix_weights: Dict[str, float],
                 encoder_len: int, decoder_len: int, micro_batch: int, seed: int = 0):
        self.sources = sources
        self.keys = list(sources.keys())
        self.p = np.array([mix_weights[k] for k in self.keys])
        self.encoder_len = encoder_len
        self.decoder_len = decoder_len
        self.total_len = encoder_len + decoder_len + 1
        self.micro_batch = micro_batch
        self.seed = seed

    def _rng(self, step: int) -> np.random.Generator:
        s = (self.seed * 1_000_003 + step * 2_654_435_761) & 0xFFFFFFFF
        return np.random.default_rng(s)

    def get_batch(self, step: int):
        rng = self._rng(step)
        B = self.micro_batch
        enc_inputs = np.empty((B, self.encoder_len), dtype=np.int64)
        dec_inputs = np.empty((B, self.decoder_len), dtype=np.int64)
        dec_targets = np.empty((B, self.decoder_len), dtype=np.int64)
        for b in range(B):
            src = rng.choice(self.keys, p=self.p)
            reader = self.sources[src]
            shard_idx = rng.choice(len(reader.arrays), p=reader.shard_weights)
            arr = reader.arrays[shard_idx]
            max_start = max(1, len(arr) - self.total_len)
            start = int(rng.integers(0, max_start))
            window = np.asarray(arr[start:start + self.total_len], dtype=np.int64)
            if window.size < self.total_len:
                pad = np.zeros(self.total_len - window.size, dtype=np.int64)
                window = np.concatenate([window, pad])
            enc_inputs[b] = window[:self.encoder_len]
            dec_seq = window[self.encoder_len:self.encoder_len + self.decoder_len]
            # Decoder input: BOS + first decoder_len-1 tokens of dec_seq
            dec_inputs[b, 0] = BOS_ID
            dec_inputs[b, 1:] = dec_seq[:-1]
            dec_targets[b] = dec_seq
        return (torch.from_numpy(enc_inputs),
                torch.from_numpy(dec_inputs),
                torch.from_numpy(dec_targets))

class FixedValLoaderSeq2Seq:
    def __init__(self, sources: Dict[str, ShardReader], encoder_len: int, decoder_len: int,
                 windows_per_source: int, batch_size: int = 8):
        self.encoder_len = encoder_len
        self.decoder_len = decoder_len
        self.total_len = encoder_len + decoder_len + 1
        self.batch_size = batch_size
        self.windows = []
        for name, reader in sources.items():
            if reader.val_arr is None:
                continue
            n = min(windows_per_source, len(reader.val_arr) // self.total_len)
            for i in range(n):
                s = i * self.total_len
                w = np.asarray(reader.val_arr[s:s + self.total_len], dtype=np.int64)
                self.windows.append((name, w))

    def __len__(self):
        return (len(self.windows) + self.batch_size - 1) // self.batch_size

    def __iter__(self):
        for i in range(0, len(self.windows), self.batch_size):
            chunk = self.windows[i:i + self.batch_size]
            B = len(chunk)
            enc_inputs = np.zeros((B, self.encoder_len), dtype=np.int64)
            dec_inputs = np.zeros((B, self.decoder_len), dtype=np.int64)
            dec_targets = np.zeros((B, self.decoder_len), dtype=np.int64)
            for j, (_, w) in enumerate(chunk):
                enc_inputs[j] = w[:self.encoder_len]
                dec_seq = w[self.encoder_len:self.encoder_len + self.decoder_len]
                dec_inputs[j, 0] = BOS_ID
                dec_inputs[j, 1:] = dec_seq[:-1]
                dec_targets[j] = dec_seq
            yield (torch.from_numpy(enc_inputs),
                   torch.from_numpy(dec_inputs),
                   torch.from_numpy(dec_targets))


In [17]:
# 6. Training Configuration and Loop
# =============================================================================
@dataclass
class TrainConfig:
    micro_batch: int = 8
    grad_accum: int = 8
    peak_lr: float = 6e-4
    min_lr: float = 6e-5
    warmup_steps: int = 300
    total_steps: int = 2000
    weight_decay: float = 0.1
    betas: Tuple[float, float] = (0.9, 0.95)
    eps: float = 1e-8
    grad_clip: float = 1.0
    eval_every: int = 250
    eval_windows: int = 128
    log_every: int = 25
    ckpt_every_minutes: float = 20
    time_budget_hours: float = 7.0
    seed: int = 1337
    output_dir: str = "runs/seq2seq_01"

def cosine_lr(step, warmup, total, peak, minimum):
    if step < warmup:
        return peak * (step + 1) / warmup
    if step >= total:
        return minimum
    prog = (step - warmup) / max(1, total - warmup)
    return minimum + 0.5 * (peak - minimum) * (1 + math.cos(math.pi * prog))

def evaluate_seq2seq(model, val_loader, device, use_fp16):
    model.eval()
    total_loss, total_tok = 0.0, 0
    with torch.no_grad():
        for enc_x, dec_x, dec_y in val_loader:
            enc_x = enc_x.to(device)
            dec_x = dec_x.to(device)
            dec_y = dec_y.to(device)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_fp16):
                logits, _ = model(enc_x, dec_x)
                loss = F.cross_entropy(logits.view(-1, logits.size(-1)),
                                       dec_y.view(-1), ignore_index=-100)
            total_loss += loss.item() * dec_y.numel()
            total_tok += dec_y.numel()
    model.train()
    return total_loss / max(1, total_tok)

def train_seq2seq(cfg: ModelConfig, tcfg: TrainConfig, train_loader, val_loader):
    out_dir = Path(tcfg.output_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    log_path = out_dir / "log.jsonl"

    model = Seq2SeqLM(cfg).to(device)
    init_weights(model, cfg)
    pc = count_params(model)
    print(f"Model params: total={pc['total']/1e6:.2f}M | emb={pc['embedding']/1e6:.2f}M | non-emb={pc['non_embedding']/1e6:.2f}M")
    assert 50_000_000 <= pc["total"] <= 300_000_000, "Param count out of range"

    opt = torch.optim.AdamW(
        model.parameters(), lr=tcfg.peak_lr, betas=tcfg.betas, eps=tcfg.eps,
        weight_decay=tcfg.weight_decay
    )
    scaler = torch.amp.GradScaler('cuda', enabled=USE_FP16)

    start_step = 0
    model.train()
    t0 = time.time()
    last_ckpt_t = time.time()
    tokens_seen = 0
    best_val = float("inf")

    for step in range(start_step, tcfg.total_steps):
        lr = cosine_lr(step, tcfg.warmup_steps, tcfg.total_steps, tcfg.peak_lr, tcfg.min_lr)
        for g in opt.param_groups:
            g["lr"] = lr

        opt.zero_grad(set_to_none=True)
        step_loss = 0.0
        for micro in range(tcfg.grad_accum):
            enc_x, dec_x, dec_y = train_loader.get_batch(step * tcfg.grad_accum + micro)
            enc_x = enc_x.to(device, non_blocking=True)
            dec_x = dec_x.to(device, non_blocking=True)
            dec_y = dec_y.to(device, non_blocking=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_FP16):
                logits, _ = model(enc_x, dec_x)
                loss = F.cross_entropy(logits.view(-1, logits.size(-1)),
                                       dec_y.view(-1), ignore_index=-100)
            scaler.scale(loss / tcfg.grad_accum).backward()
            step_loss += loss.item() / tcfg.grad_accum
            tokens_seen += dec_y.numel()

        scaler.unscale_(opt)
        gn = torch.nn.utils.clip_grad_norm_(model.parameters(), tcfg.grad_clip)
        scaler.step(opt); scaler.update()

        if step % tcfg.log_every == 0:
            dt = time.time() - t0
            tok_s = tokens_seen / max(1e-6, dt)
            print(f"step {step:5d} | loss {step_loss:.4f} | lr {lr:.2e} | gnorm {gn:.2f} | {tok_s/1000:.1f}k tok/s")
            with open(log_path, "a") as f:
                f.write(json.dumps({"step": step, "loss": step_loss, "lr": lr,
                                    "grad_norm": float(gn), "tokens_seen": tokens_seen,
                                    "elapsed_s": dt}) + "\n")

        if step > 0 and step % tcfg.eval_every == 0:
            val_loss = evaluate_seq2seq(model, val_loader, device, USE_FP16)
            ppl = math.exp(val_loss)
            print(f"  [eval] step {step} val_loss={val_loss:.4f} ppl={ppl:.2f}")
            with open(log_path, "a") as f:
                f.write(json.dumps({"step": step, "val_loss": val_loss}) + "\n")
            if val_loss < best_val:
                best_val = val_loss
                torch.save({"model": model.state_dict(), "config": asdict(cfg),
                            "step": step, "val_loss": val_loss}, out_dir / "best.pt")
            model.train()

        if (time.time() - last_ckpt_t) / 60 >= tcfg.ckpt_every_minutes:
            torch.save({"model": model.state_dict(), "opt": opt.state_dict(),
                        "scaler": scaler.state_dict(), "step": step,
                        "config": asdict(cfg), "train_config": asdict(tcfg)},
                       out_dir / "last.pt")
            last_ckpt_t = time.time()
            print(f"  [ckpt] saved at step {step}")

        if (time.time() - t0) / 3600 >= tcfg.time_budget_hours:
            print(f"Time budget reached at step {step}")
            break

    torch.save({"model": model.state_dict(), "config": asdict(cfg), "step": step},
               out_dir / "final.pt")
    return model, out_dir


In [18]:
# 7. Generation for Seq2Seq (UPDATED)
@torch.no_grad()
def generate_seq2seq(model, enc_input_ids, enc_attention_mask=None,
                     max_new_tokens=128, temperature=0.0, top_k=0, top_p=1.0,
                     repetition_penalty=1.0, no_repeat_ngram=0, first_valid_id=123):
    model.eval()
    B, dev = enc_input_ids.size(0), enc_input_ids.device
    enc_out = model.encoder(enc_input_ids, attention_mask=enc_attention_mask)

    dec_ids = torch.full((B, 1), BOS_ID, dtype=torch.long, device=dev)
    cur_pos = torch.zeros(B, 1, dtype=torch.long, device=dev)
    generated = [[] for _ in range(B)]
    kv_caches = None

    for _ in range(max_new_tokens):
        h, kv_caches = model.decoder(dec_ids, enc_out, position_ids=cur_pos,
                                     kv_caches=kv_caches,
                                     enc_attention_mask=enc_attention_mask)
        logits = model.lm_head(h[:, -1, :]).float()      # <-- THE FIX
        logits[:, :first_valid_id] = -float("inf")       # no PAD/UNK/BOS/EOS/sentinels

        for i in range(B):
            g = generated[i]
            if repetition_penalty != 1.0 and g:
                idx = torch.tensor(list(set(g)), device=dev)
                l = logits[i, idx]
                logits[i, idx] = torch.where(l > 0, l / repetition_penalty, l * repetition_penalty)
            n = no_repeat_ngram
            if n > 1 and len(g) >= n - 1:
                prefix = tuple(g[-(n - 1):])
                banned = [g[j + n - 1] for j in range(len(g) - n + 1)
                          if tuple(g[j:j + n - 1]) == prefix]
                if banned:
                    logits[i, banned] = -float("inf")

        if temperature == 0.0:
            nxt = logits.argmax(-1)
        else:
            logits = logits / temperature
            if top_k > 0:
                v, _ = torch.topk(logits, top_k, dim=-1)
                logits[logits < v[:, [-1]]] = -float("inf")
            if top_p < 1.0:
                srt, idx = torch.sort(logits, descending=True, dim=-1)
                mask = srt.softmax(-1).cumsum(-1) > top_p
                mask[:, 1:] = mask[:, :-1].clone(); mask[:, 0] = False
                srt[mask] = -float("inf")
                logits = srt.scatter(-1, idx, srt)
            nxt = torch.multinomial(logits.softmax(-1), 1).squeeze(-1)

        for i in range(B):
            generated[i].append(int(nxt[i]))
        dec_ids = nxt.unsqueeze(1)
        cur_pos = cur_pos + 1
    return generated

In [30]:
# 8. Submission Helper
# =============================================================================
def truncate_words(text: str, n: int = 128) -> str:
    words = text.split()
    return " ".join(words[:n])

def load_test_prompts_hf(max_n: int = None) -> Dict[str, str]:
    from datasets import load_dataset
    ds = load_dataset("Exploration-Lab/CS-781-Capstone", "phase_1",
                      split="test", streaming=True, token=True)
    out = {}
    for row in ds:
        for k, v in row.items():
            if isinstance(v, dict):
                out[str(k)] = v.get("text", "")
            elif isinstance(v, str):
                out[str(k)] = v
            if max_n is not None and len(out) >= max_n:
                return out
    return out

@torch.no_grad()
@torch.no_grad()
def make_submission(model, prompts, out_path, chunk_tokens=128, n_chunks=2,
                    batch_size=16, temperature=0.0, top_p=1.0, top_k=0, max_new_tokens=None,
                    repetition_penalty=1.1, no_repeat_ngram=4):
    model.eval()
    ids = list(prompts.keys())
    ctx = [tokenizer.encode(prompts[k], allow_special=False) for k in ids]
    gen = [[] for _ in ids]
    E = model.cfg.encoder_len
    order = sorted(range(len(ids)), key=lambda i: len(ctx[i]))

    for c in range(n_chunks):
        for s in range(0, len(order), batch_size):
            batch = order[s:s + batch_size]
            wins = [ctx[j][-E:] for j in batch]            # END of the context
            L = max(len(w) for w in wins)
            enc = torch.full((len(batch), L), PAD_ID, dtype=torch.long, device=device)
            msk = torch.zeros((len(batch), L), dtype=torch.long, device=device)
            for bi, w in enumerate(wins):
                enc[bi, :len(w)] = torch.tensor(w, device=device)
                msk[bi, :len(w)] = 1
            outs = generate_seq2seq(model, enc, enc_attention_mask=msk,
                                    max_new_tokens=chunk_tokens, temperature=temperature,
                                    top_k=top_k, top_p=top_p,
                                    repetition_penalty=repetition_penalty,
                                    no_repeat_ngram=no_repeat_ngram)
            for j, o in zip(batch, outs):
                gen[j] += o
                ctx[j] += o                                 # slide window for next chunk
        print(f"chunk {c+1}/{n_chunks} done")

    predictions = {ids[j]: {"text": truncate_words(tokenizer.decode(gen[j]), 128)}
                   for j in range(len(ids))}
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(json.dumps(predictions, ensure_ascii=False))
    with zipfile.ZipFile(out_path.parent / "submission.zip", "w", zipfile.ZIP_DEFLATED) as z:
        z.write(out_path, arcname="prediction.json")
    wc = [len(v["text"].split()) for v in predictions.values()]
    print(f"wrote {len(predictions)} preds | words min/mean/max = {min(wc)}/{sum(wc)/len(wc):.1f}/{max(wc)}")

In [29]:
# 9. Main Execution — FULL GPU RUN
# =============================================================================
if __name__ == "__main__":
    # Build readers
    SOURCES = ["acts", "high_courts", "supreme_court", "tribunals"]
    readers = {}
    for s in SOURCES:
        d = SHARD_DIR / s
        if (d / "meta.json").exists():
            readers[s] = ShardReader(d)
    print("Readers:", {k: v.total_tokens for k, v in readers.items()})

    # Mix weights based on token counts (temperature=1.0)
    total_tokens = sum(r.total_tokens for r in readers.values())
    mix_weights = {k: v.total_tokens / total_tokens for k, v in readers.items()}
    print("Mix weights:", mix_weights)

    # Config
    cfg = ModelConfig(
        vocab_size=VOCAB_SIZE,
        n_encoder_layers=6,
        n_decoder_layers=6,
        d_model=512,
        n_head=8,
        head_dim=64,
        encoder_len=512,
        decoder_len=128,
        max_seq_len=1024,
    )


    # GPU Training Configuration
    tcfg = TrainConfig(
        micro_batch=8,            # Fits comfortably in T4 memory
        grad_accum=8,             # Effective batch size = 64 * 640 tokens = 40,960 tokens/step
        peak_lr=6e-4,
        min_lr=6e-5,
        # warmup_steps=400,
        warmup_steps=10,
        # total_steps=7000,         # Approx 7 hours on T4 (5000 * ~5s = 7h)
        total_steps=50,         # Approx 7 hours on T4 (5000 * ~5s = 7h)
        # eval_every=500,           # Evaluates ~20 times over the run
        eval_every=10,           # Evaluates ~20 times over the run
        # log_every=50,
        log_every=5,
        ckpt_every_minutes=3,    # Saves last.pt every 20 mins
        # ckpt_every_minutes=20,    # Saves last.pt every 20 mins
        time_budget_hours=0.2,    # Hard stop at 7.5 hours to ensure submission finishes before Kaggle's 9h limit
        # time_budget_hours=9,    # Hard stop at 7.5 hours to ensure submission finishes before Kaggle's 9h limit
        output_dir=str(WORK_DIR / "runs" / "seq2seq_01"),
    )

    # Data loaders
    train_loader = Seq2SeqWindowLoader(readers, mix_weights,
                                       encoder_len=cfg.encoder_len,
                                       decoder_len=cfg.decoder_len,
                                       micro_batch=tcfg.micro_batch,
                                       seed=SEED)
    val_loader = FixedValLoaderSeq2Seq(readers,
                                       encoder_len=cfg.encoder_len,
                                       decoder_len=cfg.decoder_len,
                                       windows_per_source=64,
                                       batch_size=8)

    # Train
    print("\nStarting training...")
    model, run_dir = train_seq2seq(cfg, tcfg, train_loader, val_loader)

    # Generate submission from best checkpoint
    print("\nLoading best checkpoint for submission...")
    ckpt = torch.load(run_dir / "best.pt", map_location=device)
    model.load_state_dict(ckpt["model"])
    model = model.to(device).eval()
    print(f"Loaded step {ckpt['step']}, val_loss {ckpt['val_loss']:.4f}")

    prompts = load_test_prompts_hf()
    print(f"Loaded {len(prompts)} test prompts")
    
    # Generate predictions
    make_submission(
        model, prompts,
        out_path=WORK_DIR / "submission_seq2seq" / "prediction.json",
        max_new_tokens=128,
        batch_size=16,            # Can increase batch size for inference on GPU
        temperature=0.0,
        repetition_penalty=1.1,
    )

    print("\nDone! Submit the generated submission.zip to Codabench.")

Readers: {'acts': 20010586, 'high_courts': 65000132, 'supreme_court': 25002938, 'tribunals': 40000225}
Mix weights: {'acts': 0.1333915626114626, 'high_courts': 0.4332941162958113, 'supreme_court': 0.1666708296147608, 'tribunals': 0.2666434914779653}

Starting training...
Model params: total=61.23M | emb=16.38M | non-emb=44.84M
step     0 | loss 10.4817 | lr 6.00e-05 | gnorm 9.83 | 2.9k tok/s
step     5 | loss 9.1089 | lr 3.60e-04 | gnorm 2.43 | 3.1k tok/s
step    10 | loss 7.8126 | lr 6.00e-04 | gnorm 2.04 | 3.1k tok/s
  [eval] step 10 val_loss=7.5216 ppl=1847.54
step    15 | loss 6.6760 | lr 5.79e-04 | gnorm 1.13 | 2.9k tok/s
step    20 | loss 6.5334 | lr 5.21e-04 | gnorm 0.56 | 2.9k tok/s
  [eval] step 20 val_loss=6.3671 ppl=582.39
step    25 | loss 6.3215 | lr 4.33e-04 | gnorm 1.12 | 2.8k tok/s
step    30 | loss 6.1611 | lr 3.30e-04 | gnorm 0.72 | 2.9k tok/s
  [eval] step 30 val_loss=6.1759 ppl=481.00
step    35 | loss 6.2882 | lr 2.27e-04 | gnorm 1.69 | 2.8k tok/s
step    40 | loss

In [33]:
# 1. Load the trained model from your uploaded files
# If you uploaded it via "Add Data", the path will be /kaggle/input/your-dataset-name/best.pt
ckpt_path = "/kaggle/input/datasets/ascyrax/ds-caps-enc-dec-01-inference/best.pt" # Update this to your actual path!

print("Loading best checkpoint...")
ckpt = torch.load(ckpt_path, map_location=device)
# model = Seq2SeqLM(ckpt["config"]) 

# ✅ CORRECT (what it should be)
cfg_loaded = ModelConfig(**ckpt["config"])
model = Seq2SeqLM(cfg_loaded)

model.load_state_dict(ckpt["model"])
model = model.to(device).eval()
print(f"Loaded step {ckpt['step']}, val_loss {ckpt['val_loss']:.4f}")

# 2. Generate the fixed submission
prompts = load_test_prompts_hf()

# Generate predictions
make_submission(
    model, prompts,
    out_path=WORK_DIR / "submission_seq2seq" / "prediction.json",
    max_new_tokens=128,
    batch_size=16,            # Can increase batch size for inference on GPU
    temperature=0.0,
    repetition_penalty=1.1,
)



Loading best checkpoint...
Loaded step 10500, val_loss 2.4023
chunk 1/2 done
chunk 2/2 done
wrote 1000 preds | words min/mean/max = 61/127.6/128


In [34]:
# Cell 10: Verify the quick-test model actually generates text
import json
from pathlib import Path

# Load the best model from the quick test
quick_dir = WORK_DIR / "runs" / "seq2seq_01"
ckpt = torch.load(quick_dir / "best.pt", map_location=device)
model.load_state_dict(ckpt["model"])
model = model.to(device).eval()

# Grab one prompt
prompts = load_test_prompts_hf(max_n=1)
prompt_text = list(prompts.values())[0]
prompt_ids = tokenizer.encode(prompt_text, allow_special=False)[-512:]

enc_inputs = torch.tensor([prompt_ids], device=device)
enc_mask = torch.ones_like(enc_inputs)

outs = generate_seq2seq(model, enc_inputs, enc_attention_mask=enc_mask, 
                        max_new_tokens=50, temperature=0.0)
generated_text = tokenizer.decode(outs[0])

print("--- PROMPT ---")
print(prompt_text[:200] + "...")
print("\n--- GENERATED ---")
print(generated_text)
print("----------------")

--- PROMPT ---
The issue involved in these petitions is as to whether the acquisition proceedings can be challenged at a belated stage. The issue is no more res integra as the issue has been considered by this Court...

--- GENERATED ---
.

















































----------------


In [35]:
# Cell 11: Validate the submission JSON and Zip structure
sub_path = WORK_DIR / "submission_seq2seq" / "prediction.json"
zip_path = WORK_DIR / "submission_seq2seq" / "submission.zip"

if sub_path.exists():
    sub = json.loads(sub_path.read_text())
    print(f"Total predictions: {len(sub)}")
    
    # Check word counts
    wc = [len(v["text"].split()) for v in sub.values()]
    print(f"Word counts (min/mean/max): {min(wc)} / {sum(wc)/len(wc):.1f} / {max(wc)}")
    
    # Check Zip structure
    with zipfile.ZipFile(zip_path, 'r') as z:
        print(f"Zip contents: {z.namelist()}")
        assert 'prediction.json' in z.namelist(), "prediction.json must be at the root of the zip!"
    print("✅ Submission format is valid.")
else:
    print("❌ Submission file not found. Did the training complete?")

Total predictions: 1000
Word counts (min/mean/max): 61 / 127.6 / 128
Zip contents: ['prediction.json']
✅ Submission format is valid.
